# ⚙️ Setup Universal — Módulo 4 Deep Learning
**Diplomado Superior en Redes Neuronales Artificiales y Deep Learning**

Este notebook detecta automáticamente el ambiente (Google Colab o local)  
y configura todo sin que tengas que cambiar nada a mano.

---
### ¿Cómo usarlo?
1. Corre la **Celda 1** primero — detecta el ambiente y configura rutas
2. Corre la **Celda 2** — instala/verifica librerías
3. Corre la **Celda 3** — verifica GPU
4. Copia estas celdas al inicio de cualquier notebook del módulo ✓

---
## 📍 Celda 1 — Detección de ambiente y rutas

In [ ]:
# ══════════════════════════════════════════════════════════════
# SETUP UNIVERSAL — corre esto PRIMERO en cualquier notebook
# Funciona igual en Google Colab y en VSCode/Jupyter local
# NOTA: También puedes usar setup_completo() de la librería:
#   from modulo4_libreria import *
#   INFO = setup_completo()
# ══════════════════════════════════════════════════════════════
import os, sys

# ── Detectar ambiente ─────────────────────────────────────────
EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    print("🌐 Ambiente: Google Colab")
    
    # Montar Google Drive (para guardar modelos y datasets)
    # Solo se pide permiso la primera vez
    from google.colab import drive
    drive.mount('/content/drive')
    
    # ── Rutas en Colab ────────────────────────────────────────
    # /content/       → carpeta temporal (se borra al cerrar sesión)
    # /content/drive/ → tu Google Drive (persiste)
    RUTA_BASE    = '/content/drive/MyDrive/Diplomado/Modulo4'
    RUTA_DATOS   = os.path.join(RUTA_BASE, 'datos')
    RUTA_MODELOS = os.path.join(RUTA_BASE, 'modelos')
    RUTA_OUTPUTS = os.path.join(RUTA_BASE, 'outputs')
    
else:
    print("🖥️  Ambiente: Local (VSCode / Jupyter)")
    
    # ── Rutas locales ─────────────────────────────────────────
    # Usa rutas relativas al notebook actual → funciona en cualquier PC
    RUTA_BASE    = os.path.abspath(os.path.dirname('__file__') if '__file__' in dir() else '.')
    RUTA_DATOS   = os.path.join(RUTA_BASE, 'datos')
    RUTA_MODELOS = os.path.join(RUTA_BASE, 'modelos')
    RUTA_OUTPUTS = os.path.join(RUTA_BASE, 'outputs')

# ── Crear carpetas si no existen ──────────────────────────────
for ruta in [RUTA_DATOS, RUTA_MODELOS, RUTA_OUTPUTS]:
    os.makedirs(ruta, exist_ok=True)

print(f"\n📁 Rutas configuradas:")
print(f"   Base    : {RUTA_BASE}")
print(f"   Datos   : {RUTA_DATOS}")
print(f"   Modelos : {RUTA_MODELOS}")
print(f"   Outputs : {RUTA_OUTPUTS}")

---
## 📦 Celda 2 — Instalación y verificación de librerías

In [ ]:
# ── En Colab: instala lo que no viene por defecto ─────────────
# En local: solo verifica que estén instaladas

if EN_COLAB:
    print("📦 Instalando librerías en Colab...")
    # TF, numpy, matplotlib, sklearn ya vienen en Colab
    # Agrega aquí lo que necesites extra:
    # !pip install -q transformers datasets  # para Transformers/HuggingFace
    # !pip install -q torch torchvision      # si el profe usa PyTorch
    print("✓ Librerías base ya incluidas en Colab")
else:
    print("🖥️  Local: verificando librerías...")

# ── Intento: cargar modulo4_libreria ──────────────────────────
try:
    import sys, os
    sys.path.insert(0, os.path.abspath(os.path.join('..')))
    from modulo4_libreria import *
    print('✅ modulo4_libreria cargada')
except ImportError:
    pass

# ── Imports estándar del módulo 4 ─────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ── Semillas de reproducibilidad ──────────────────────────────
# Esto garantiza que tus resultados sean iguales cada que corres el notebook
SEMILLA = 42
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)

# ── Configuración de visualización ───────────────────────────
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid', palette='muted')

print(f"\n✅ Librerías cargadas:")
print(f"   NumPy       {np.__version__}")
print(f"   Pandas      {pd.__version__}")
print(f"   TensorFlow  {tf.__version__}")
print(f"   Matplotlib  {matplotlib.__version__}")

---
## 🎮 Celda 3 — Verificación de GPU

In [ ]:
# ── Verificar GPU disponible ──────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"🎮 GPU detectada: {len(gpus)} dispositivo(s)")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
    
    if EN_COLAB:
        # En Colab: ver detalles de la GPU asignada
        # (puede ser T4, A100, L4 dependiendo del plan)
        gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
        print(f"   Modelo : {gpu_info[0]}")
    else:
        # Local: info de la GPU
        try:
            import subprocess
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                capture_output=True, text=True
            )
            if result.returncode == 0:
                print(f"   Modelo : {result.stdout.strip()}")
        except:
            pass
    
    # Permitir crecimiento de memoria (evita que TF acapare toda la VRAM)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("   Modo memoria dinámica: activado ✓")

else:
    print("⚠️  No se detectó GPU — se usará CPU")
    if EN_COLAB:
        print("   En Colab: ve a Entorno de ejecución → Cambiar tipo de entorno → GPU")
    else:
        print("   Local: verifica que CUDA y cuDNN estén instalados")

print(f"\n🔧 Dispositivo activo: {tf.test.gpu_device_name() or 'CPU'}")

---
## 💾 Celda 4 — Helpers de guardado/carga (adaptativos)
Funciones para guardar modelos y resultados que funcionan igual en Colab y local.

In [ ]:
# MACHOTE — Guardar y cargar modelos de forma portable

def guardar_modelo(modelo, nombre):
    """
    Guarda el modelo en RUTA_MODELOS.
    En Colab: se guarda en Drive (persiste después de cerrar)
    En local: se guarda junto al notebook
    """
    ruta = os.path.join(RUTA_MODELOS, f'{nombre}.keras')
    modelo.save(ruta)
    print(f"✅ Modelo guardado en: {ruta}")
    return ruta

def cargar_modelo(nombre):
    """
    Carga un modelo guardado previamente.
    """
    ruta = os.path.join(RUTA_MODELOS, f'{nombre}.keras')
    if os.path.exists(ruta):
        modelo = keras.models.load_model(ruta)
        print(f"✅ Modelo cargado desde: {ruta}")
        return modelo
    else:
        print(f"❌ No se encontró el modelo en: {ruta}")
        return None

def guardar_figura(nombre, dpi=150):
    """
    Guarda la figura actual de matplotlib en RUTA_OUTPUTS.
    Útil para incluir gráficas en el reporte sin capturas de pantalla.
    """
    ruta = os.path.join(RUTA_OUTPUTS, f'{nombre}.png')
    plt.savefig(ruta, dpi=dpi, bbox_inches='tight')
    print(f"✅ Figura guardada en: {ruta}")

# Estas funciones también están disponibles en modulo4_libreria
# como guardar_modelo(), cargar_modelo(), guardar_figura()
print("💾 Helpers de guardado listos ✓")

---
## 📤 Celda 5 — Subir archivos (solo Colab)
Si necesitas subir un dataset desde tu compu a Colab.

In [ ]:
# ── Subir archivos a Colab (solo funciona en Colab) ───────────
# En local esta celda no hace nada (no rompe el notebook)

if EN_COLAB:
    from google.colab import files
    
    # OPCIÓN A: subir manualmente desde tu compu
    # uploaded = files.upload()  # abre un selector de archivos
    # for nombre_archivo in uploaded.keys():
    #     print(f"Subido: {nombre_archivo}")
    
    # OPCIÓN B: descargar dataset desde URL directamente en Colab
    # !wget -q URL_DEL_DATASET -O {RUTA_DATOS}/dataset.csv
    
    # OPCIÓN C: usar datasets de Keras (no requieren descarga manual)
    # (X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
    
    print("📤 Opciones de carga disponibles — descomenta la que necesites")
else:
    print("🖥️  Local: carga los datos directamente con pd.read_csv() o keras.datasets")
    print(f"   Ruta sugerida para datos: {RUTA_DATOS}")

---
## 📋 Resumen del ambiente
Corre esta celda al final para confirmar que todo está OK antes de empezar a trabajar.

In [ ]:
print("=" * 55)
print("  RESUMEN DE AMBIENTE — Módulo 4 Deep Learning")
print("=" * 55)
print(f"  Ambiente    : {'Google Colab' if EN_COLAB else 'Local (VSCode/Jupyter)'}")
print(f"  TensorFlow  : {tf.__version__}")
print(f"  Python      : {sys.version.split()[0]}")
print(f"  GPU         : {tf.test.gpu_device_name() or 'No detectada (CPU)'}")
print(f"  Ruta base   : {RUTA_BASE}")
print(f"  Semilla     : {SEMILLA}")
print("=" * 55)
print("  ✅ Listo para trabajar")
print("=" * 55)

---
## 📖 Guía rápida: Colab para quien viene de VSCode

| VSCode | Google Colab | Notas |
|---|---|---|
| Abres el archivo `.ipynb` | Subes a Drive o abres desde GitHub | Colab guarda automáticamente en Drive |
| Ctrl+Enter = corre celda | Shift+Enter o ▶ = corre celda | Igual |
| Terminal integrada | `!comando` en celda de código | `!pip install`, `!ls`, `!wget` |
| Extensión Python / kernel local | Runtime asignado por Google | Puedes cambiar GPU en Entorno de ejecución |
| Archivos en tu disco | Archivos en `/content/` (temporal) | Monta Drive para que persistan |
| Variables persisten mientras corre el kernel | Variables se pierden si la sesión se desconecta | Guarda modelos en Drive |
| Puedes usar tu GPU (RTX 4060) | GPU de Google (T4 gratis, A100 Pro) | En casa, local es mejor con tu 4060 |

### Tips para Colab:
- **Sesión inactiva:** Colab desconecta después de ~90 min sin actividad. Guarda modelos en Drive frecuentemente.
- **Cambiar GPU:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU
- **Ver archivos:** ícono de carpeta en el panel izquierdo
- **Montar Drive:** ya está en la Celda 1 de arriba — solo acepta el permiso

---
## 📌 Compatibilidad con modulo4_libreria

Este notebook **setup_universal** y **modulo4_libreria.py** son complementarios:

| Recurso | Cuándo usarlo |
|---|---|
| **setup_universal.ipynb** | Cuando necesitas configurar el ambiente desde cero (Colab o local) |
| **modulo4_libreria.py** | Funciones reutilizables listas para importar en cualquier notebook |

### Ventajas de usar modulo4_libreria:
- `setup_completo()` → hace el trabajo de Celdas 1-3 automáticamente
- `crear_mlp()`, `crear_cnn()`, `crear_lstm_texto()` → modelos predefinidos
- `compilar_y_entrenar()` → compila con loss automático y early stopping
- `evaluar_clasificacion()` → reporte completo con matriz de confusión
- `guardar_modelo()`, `cargar_modelo()`, `guardar_figura()` → helpers de guardado

> 💡 **Recomendación:** Para notebooks nuevos, usa `from modulo4_libreria import *` + `setup_completo()` como punto de partida.